# Phase 2: Deep Learning Pipeline for Skin Condition Classification

This notebook demonstrates the Phase 2 deep learning approach using an **EfficientNet-B3** backbone with a **Convolutional Block Attention Module (CBAM)**.

### Running on Google Colab (Recommended)
For significantly faster training, upload this script (or convert to `.ipynb` via Jupytext) to Google Colab and enable a hardware accelerator:
1. Go to **Runtime > Change runtime type**
2. Select **T4 GPU** or **TPU v5e-1**
3. Ensure the `Multi-Class Skin Condition Image Dataset (MSC-6)` is uploaded or linked in Colab's file system, and update the `data_dir` path below accordingly.

In [2]:
# Install the required libraries on the Colab server
!pip install timm grad-cam joblib scikit-image


## 1. Setup and Configuration

In [6]:
import sys
import os
from pathlib import Path

# First, check if we are on Google Colab and the Drive is mounted
colab_path = Path("/content/drive/MyDrive/Hybrid-Dermatologist")

if colab_path.exists():
    project_root = colab_path
else:
    # Otherwise, assume we are on a local machine or terminal server
    project_root = Path(os.getcwd()).resolve()
    # If we started inside the notebooks folder, go up 2 levels to the root
    if project_root.name == "phase2":
        project_root = project_root.parents[1]

project_path = str(project_root)

# Verify the path exists and set it up
if project_root.exists():
    os.chdir(project_path)
    if project_path not in sys.path:
        sys.path.insert(0, project_path)
    print(f"✅ Success! Working directory set to: {os.getcwd()}")
    
    # Import required components
    from src.skin_analysis.phase2 import Phase2Config, run_phase2_pipeline
    from src.skin_analysis.phase2.train import detect_device
    print("✅ Imports successful!")
else:
    print(f"❌ Error: Could not find project root at {project_path}")


✅ Success! Working directory set to: /content/drive/MyDrive/Hybrid-Dermatologist
✅ Imports successful!


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
# Define configuration

data_dir = project_root / "data" / "raw" / "Multi-Class Skin Condition Image Dataset (MSC-6)"
output_dir = project_root / "outputs" / "phase2_deep_learning"

cfg = Phase2Config(
    data_dir=data_dir,
    output_dir=output_dir,
    batch_size=32,       # Can increase to 64 if on Colab T4 GPU
    epochs_stage1=15,    # Head-only warmup
    epochs_stage2=30,    # Full fine-tuning
)

device = detect_device()
print(f"Detected training device: {device}")
if device.type == "cpu":
    print("WARNING: Training on CPU will be very slow. Consider using Google Colab with a GPU/TPU.")

Detected training device: cuda


## 2. Data Loading & Augmentation Preview

We use **MixUp** and **RandAugment** to regularise the model, along with a **WeightedRandomSampler** to handle the 13x class imbalance between Eczema and Normal.

In [9]:
import matplotlib.pyplot as plt
from src.skin_analysis.phase2.augment import build_dataloaders, visualize_augmented_samples

print("Loading datasets...")
train_loader, val_loader, train_dataset, val_dataset = build_dataloaders(cfg)

# Generate and display augmented samples
sample_path = cfg.output_dir / "augmented_samples.png"
cfg.output_dir.mkdir(parents=True, exist_ok=True)
visualize_augmented_samples(train_dataset, cfg.display_names, sample_path)

img = plt.imread(sample_path)
plt.figure(figsize=(12, 12))
plt.imshow(img)
plt.axis("off")
plt.show()

Loading datasets...


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


## 3. Model Architecture

**EfficientNet-B3** (12M params) provides the feature extraction backbone, while the **CBAM** block provides spatial and channel attention before the classifier head.

In [ ]:
from src.skin_analysis.phase2.model import EfficientNetB3CBAM

model = EfficientNetB3CBAM(
    num_classes=cfg.num_classes,
    hidden_dim=cfg.hidden_dim,
    dropout=cfg.dropout,
    pretrained=True
)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:103: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Total parameters: 11,781,264
Trainable parameters: 11,781,264


: 

## 4. Full Pipeline Execution

This runs the complete Phase 2 pipeline:
1. **Stage 1 Training**: Freezes the backbone and trains only the head for 15 epochs.
2. **Stage 2 Training**: Unfreezes the last 3 EfficientNet blocks for fine-tuning.
3. **Evaluation**: Computes metrics and confusion matrix on the validation set.
4. **Grad-CAM**: Generates attention heatmaps for visual explanation.

*(Uncomment the cell below to run the full pipeline. This may take 20-30 minutes on an MPS device or T4 GPU.)*

In [11]:
print("Starting Phase 2 Pipeline...")
run_phase2_pipeline(cfg, device_override=str(device), run_gradcam=True, run_ablation=False)


Starting Phase 2 Pipeline...
  Phase 2: EfficientNet-B3 + CBAM — Skin Condition Classification
  Backbone     : efficientnet_b3
  Image size   : 300×300
  Batch size   : 32
  Stage 1      : 15 epochs, lr=0.001
  Stage 2      : 30 epochs, lr_bb=2e-05, lr_head=0.0001
  MixUp α      : 0.2
  RandAugment  : M=9, N=2
  Device       : cuda
  Data dir     : /content/drive/MyDrive/Hybrid-Dermatologist/data/raw/Multi-Class Skin Condition Image Dataset (MSC-6)
  Output dir   : /content/drive/MyDrive/Hybrid-Dermatologist/outputs/phase2_deep_learning

╔══ Loading datasets ══╗
  Train: 7879 images
  Val:   944 images
            acne:  1000
      dark_spots:   996
          eczema:  2883
          normal:  1000
         rosacea:  1000
        wrinkles:  1000

  Generating augmented samples visualisation…

╔══ Building model ══╗
  Total params     : 11,781,264
  Trainable params : 11,781,264

╔══ Stage 1: Training CBAM + classifier head (backbone frozen) ══╗

  Epoch 1/15  (lr=0.001000)


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:432: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()


  train:   0%|          | 0/246 [00:00<?, ?it/s]

: 

: 

## 5. Results & Visualization

Assuming the pipeline has been run, we can load and display the generated artifacts.

In [ ]:
def show_image(path_str):
    path = Path(path_str)
    if path.exists():
        img = plt.imread(path)
        plt.figure(figsize=(12, 10))
        plt.imshow(img)
        plt.axis("off")
        plt.show()
    else:
        print(f"Image not found: {path}")

### Learning Curves

In [ ]:
show_image(cfg.output_dir / "learning_curves.png")

### Confusion Matrix

In [ ]:
show_image(cfg.output_dir / "confusion_matrix_phase2.png")

### Baseline Comparison (Phase 1 RF vs Phase 2)

In [ ]:
import pandas as pd

comparison_path = cfg.output_dir / "baseline_comparison.csv"
if comparison_path.exists():
    df_comp = pd.read_csv(comparison_path)
    display(df_comp)
else:
    print("Baseline comparison not found. Run the pipeline first.")

### Grad-CAM Explanations

These heatmaps show *where* the model is looking when making a prediction. Notice how the attention focuses tightly on acne lesions, addressing the locality failure mode from Phase 1.

In [ ]:
show_image(cfg.output_dir / "gradcam_summary.png")

### Ablation Study

Compare the main model against variants without CBAM or fine-tuning.
*(Requires running the pipeline with `run_ablation=True`)*

In [ ]:
ablation_path = cfg.output_dir / "ablation_study.csv"
if ablation_path.exists():
    df_ablation = pd.read_csv(ablation_path)
    display(df_ablation)
else:
    print("Ablation study not found. Run the pipeline with run_ablation=True.")